# Лабораторная работа 2. Правила вместо понимания: машина Тьюринга, ELIZA и тест Тьюринга

In [ ]:
student = ""   # Фамилия Имя Отчество
group = ""     # номер группы
print(student, group)

**Основные ссылки.** Полный перечень ресурсов приведен в конце ноутбука.

| Ресурс | Адрес |
|---|---|
| Turing A. M. On Computable Numbers, with an Application to the Entscheidungsproblem. Proc. London Math. Soc., 1937, s2-42, 230–265 | https://doi.org/10.1112/plms/s2-42.1.230 |
| Turing A. M. Computing Machinery and Intelligence. Mind, 1950, 59(236), 433–460 | https://doi.org/10.1093/mind/LIX.236.433 |
| Weizenbaum J. ELIZA – a computer program for the study of natural language communication between man and machine. Communications of the ACM, 1966, 9(1), 36–45 | https://doi.org/10.1145/365153.365168 |

## 0. Устройство работы

**Цель.** Подготовить рабочую папку и понять порядок выполнения.

Работа состоит из трех частей. В каждой части приведены постановка задачи, требования к результату и блоки с заданиями. 

В блоках с комментарием `# ЗАДАНИЕ` необходимо написать код по формальному описанию, а в блоках с данными заполнить таблицы правил, сценарии и списки.

| Часть | Что реализуется |
|---|---|
| 1. Машина Тьюринга | интерпретатор машины и три таблицы правил |
| 2. ELIZA | программа ELIZA по формальному описанию и собственный сценарий |
| 3. Тест Тьюринга | протокол эксперимента, сбор ответов трех источников, автоматический судья |

**Порядок работы в блоке.** Каждый блок содержит описание интерфейса (что принимает функция, что возвращает), ячейку с заданием и проверочную ячейку. 

1) Проверочные ячейки выводят «OK» или «ОШИБКА» с ожидаемым и полученным значениями. 
2) Блоки выполняются сверху вниз. 
3) Реализацию каждой части необходимо разбить на отдельные функции, как показано в блоках, и снабдить комментариями.

**Запуск на произвольных данных.** В конце каждой части есть ячейка запуска на произвольных данных: (1) машина Тьюринга запускается на любой ленте, (2) ELIZA отвечает на любую реплику, (3) судья оценивает любую пару «вопрос – ответ». Эти ячейки используются для самопроверки.

**Пояснения.** В конце каждой части расположена ячейка «Пояснение»: в ней своими словами записывается, как устроена реализация и почему программа выводит именно такой результат.

**0.1. Заполнить сведения о студенте (первая ячейка), подготовить папку данных, вычислить код автора и определить вспомогательные функции проверки.**

In [ ]:
import sys, os, re, csv, random, hashlib
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Рабочая папка: папка с ноутбуком (если ноутбук лежит в подпапке notebooks, то ее родитель)
cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
os.chdir(ROOT)
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)

if not student.strip() or not group.strip():
    print("Заполнить сведения о студенте в первой ячейке и выполнить ее заново")

# Код автора: первые 8 символов хеша от фамилии и группы; используется в именах файлов
AUTHOR = hashlib.sha256(f"{student.strip()}|{group.strip()}".encode("utf-8")).hexdigest()[:8]

# Вспомогательные функции проверочных ячеек
CHECKS = []

def start_checks():
    CHECKS.clear()

def check(label, got, expected):
    ok = got == expected
    CHECKS.append(ok)
    print(("OK       " if ok else "ОШИБКА   ") + label)
    if not ok:
        print("   ожидалось:", repr(expected))
        print("   получено: ", repr(got))
    return ok

def safe(function, *args, **kwargs):
    # Вызывает функцию и возвращает результат; исключение возвращается как текст, чтобы проверки не прерывались
    try:
        return function(*args, **kwargs)
    except Exception as error:
        return f"Исключение {type(error).__name__}: {error}"

def raises(label, exception, function, *args, **kwargs):
    # Проверяет, что функция выбрасывает указанное исключение
    try:
        function(*args, **kwargs)
    except exception:
        return check(label, True, True)
    except Exception as error:
        return check(label, f"другое исключение: {type(error).__name__}", f"исключение {exception.__name__}")
    return check(label, "исключение не возникло", f"исключение {exception.__name__}")

def finish_checks():
    print(f"\nПроверок пройдено: {sum(CHECKS)} из {len(CHECKS)}")

print("Python:", sys.version.split()[0])
print("Рабочая папка:", ROOT)
print("Папка данных:", DATA)
print("Код автора:", AUTHOR)

## Часть 1. Машина Тьюринга

**Цель.** Реализовать интерпретатор машины Тьюринга и составить для нее три программы (таблицы правил).

**Зачем это нужно.** В теме 2 мы поговорили о правилах, которые заменяют понимание. Машина Тьюринга – исходная модель такого подхода: любое вычисление сводится к конечному набору правил вида «если в таком-то состоянии под головкой такой-то символ, то записать символ, сдвинуться и перейти в другое состояние». Ту же схему «условие – действие» использует ELIZA в части 2.

**Определение.**

| Элемент | Описание |
|---|---|
| Лента | бесконечная в обе стороны последовательность ячеек, в каждой один символ; пустая ячейка содержит символ `_` |
| Головка | указывает на одну ячейку ленты; перед началом работы стоит на первом символе входного слова (позиция 0) |
| Состояние | одно из конечного набора; начальное состояние `q0`, конечное `stop` |
| Таблица правил | словарь `{(состояние, символ): (новый_символ, движение, новое_состояние)}`; движение: `L` – влево, `R` – вправо, `N` – на месте |
| Шаг | прочитать символ под головкой, найти правило, записать новый символ, сдвинуть головку, сменить состояние |
| Нормальная остановка | машина перешла в состояние `stop` |
| Аварийная остановка | для пары (состояние, символ) нет правила (исключение `ValueError`) либо число шагов превысило `max_steps` (исключение `RuntimeError`) |
| Результат | слово на ленте после остановки: от крайней левой до крайней правой непустой ячейки |

**Требования к результату.**

| № | Требование |
|---|---|
| 1 | Реализованы функции `make_tape`, `tape_to_string`, `step`, `run_turing` с указанными интерфейсами; все проверки пройдены |
| 2 | `run_turing` возвращает трассировку: список конфигураций «шаг, состояние, позиция головки, лента» |
| 3 | Составлены таблицы `INVERT`, `ADD`, `INC`; проверки на всех тестовых словах пройдены |
| 4 | Выведена трассировка работы `INC` на слове `1011` |
| 5 | Заполнено пояснение к части 1 |

**1.1. Выполнить ячейку с обозначениями.**

In [ ]:
BLANK = "_"                         # пустая ячейка
MOVES = {"L": -1, "R": 1, "N": 0}   # движение головки: изменение позиции
START = "q0"                        # начальное состояние
STOP = "stop"                       # конечное состояние
print("Обозначения определены")

**1.2. Реализовать функцию `make_tape(word)`.**

1) Вход: слово (строка). 
2) Выход: словарь `{позиция: символ}`. 
3) Первый символ слова находится в позиции 0, следующий – в позиции 1 и так далее. 
4) Для пустого слова возвращается пустой словарь.

In [ ]:
def make_tape(word):
    # ЗАДАНИЕ: создать ленту по слову
    return None

In [ ]:
start_checks()
check("make_tape('101')", safe(make_tape, "101"), {0: "1", 1: "0", 2: "1"})
check("make_tape('')", safe(make_tape, ""), {})
finish_checks()

**1.3. Реализовать функцию `tape_to_string(tape)`.**

1) Вход: лента-словарь. 
2) Выход: строка из символов от крайней левой до крайней правой непустой ячейки. 
3) Позиции, которых нет в словаре, считаются пустыми и в середине строки записываются символом `_`. 
4) Пустые ячейки по краям в строку не входят. 
5) Если непустых ячеек нет, возвращается пустая строка.

In [ ]:
def tape_to_string(tape):
    # ЗАДАНИЕ: преобразовать ленту в строку
    return None

In [ ]:
start_checks()
check("пропуск в середине", safe(tape_to_string, {0: "1", 2: "1"}), "1_1")
check("сдвиг начала", safe(tape_to_string, {3: "0", 4: "1"}), "01")
check("пустая лента", safe(tape_to_string, {}), "")
check("пустые края не входят", safe(tape_to_string, {0: "_", 1: "1", 2: "_"}), "1")
check("отрицательные позиции", safe(tape_to_string, {-1: "1", 0: "0"}), "10")
finish_checks()

**1.4. Реализовать функцию `step(table, tape, state, head)`.**

1) Выполняет один шаг машины. 
2) Вход: таблица правил, лента (изменяется на месте), текущее состояние, позиция головки. 
3) Выход: пара `(новое_состояние, новая_позиция_головки)`. 
4) Символ под головкой читается из ленты. Если позиции нет в ленте, символ равен `BLANK`. 
5) По найденному правилу в ячейку под головкой записывается новый символ, головка сдвигается согласно `MOVES`. 
6) Если в таблице нет правила для пары (состояние, символ), выводится `ValueError`.

In [ ]:
def step(table, tape, state, head):
    # ЗАДАНИЕ: выполнить один шаг машины
    return None

In [ ]:
start_checks()
demo_table = {("q0", "1"): ("0", "R", "q1")}
demo_tape = make_tape("11")
check("step: состояние и позиция", safe(step, demo_table, demo_tape, "q0", 0), ("q1", 1))
check("step: запись символа", safe(tape_to_string, demo_tape), "01")
raises("step: нет правила", ValueError, step, demo_table, demo_tape, "q9", 0)
check("step: движение влево", safe(step, {("q0", "_"): ("1", "L", "q0")}, {}, "q0", 5), ("q0", 4))
finish_checks()

**1.5. Реализовать функцию `run_turing(table, word, max_steps=1000)`.**

1) Запускает машину на слове. 
2) Выход: тройка `(результат, число_шагов, трассировка)`. 
3) Результат – слово на ленте после нормальной остановки (`tape_to_string`). 
4) Трассировка – список кортежей `(шаг, состояние, позиция_головки, лента_строкой)`: первая запись описывает начальную конфигурацию (шаг 0), далее по одной записи после каждого выполненного шага. 
5) Работа заканчивается, когда состояние стало `STOP`. 
6) Если число выполненных шагов достигло `max_steps`, а состояние не `STOP`, выводится `RuntimeError`. 
7) Функция использует `make_tape`, `step` и `tape_to_string`.

In [ ]:
def run_turing(table, word, max_steps=1000):
    # ЗАДАНИЕ: запустить машину на слове
    return None

In [ ]:
start_checks()
ERASE = {("q0", "1"): ("_", "R", "q0"), ("q0", "_"): ("_", "N", "stop")}
result = safe(run_turing, ERASE, "111")
check("run_turing: результат", result[0] if isinstance(result, tuple) else result, "")
check("run_turing: число шагов", result[1] if isinstance(result, tuple) else result, 4)
check("run_turing: длина трассировки", len(result[2]) if isinstance(result, tuple) else result, 5)
check("run_turing: начальная конфигурация", result[2][0] if isinstance(result, tuple) else result, (0, "q0", 0, "111"))
check("run_turing: конечная конфигурация", result[2][-1] if isinstance(result, tuple) else result, (4, "stop", 3, ""))
raises("run_turing: бесконечная работа", RuntimeError, run_turing, {("q0", "_"): ("_", "N", "q0")}, "", 50)
raises("run_turing: нет правила", ValueError, run_turing, {}, "1")
finish_checks()

**1.6. Составить таблицу правил `INVERT`: инверсия двоичного слова.**

1) Вход: слово из символов `0` и `1`. 
2) Выход: слово той же длины, в котором каждый символ заменен на противоположный. 
3) Пустое слово остается пустым. 
4) Головка в начале стоит на первом символе.

In [ ]:
INVERT = {
    # ЗАДАНИЕ: добавить правила вида ("состояние", "символ"): ("новый символ", "движение", "новое состояние")
}

**1.7. Составить таблицу правил `ADD`: сложение в унарной записи.**

1) Вход: слово вида `1…1+1…1` (число палочек до и после знака `+` может быть нулевым). 
2) Выход: слово из палочек, число которых равно сумме. 

Примеры: `1+1` дает `11`; `111+11` дает `11111`; `+1` дает `1`; `+` дает пустое слово.

In [ ]:
ADD = {
    # ЗАДАНИЕ: составить таблицу правил
}

**1.8. Составить таблицу правил `INC`: увеличение двоичного числа на единицу.**

1) Вход: двоичное число (старший разряд слева). 
2) Выход: число, увеличенное на 1. 

Примеры: `0` дает `1`; `1011` дает `1100`; `111` дает `1000` (результат может быть длиннее входа: лента бесконечна влево).

In [ ]:
INC = {
    # ЗАДАНИЕ: составить таблицу правил
}

**1.9. Проверить три таблицы на тестовых словах.**

In [ ]:
TESTS = {
    "INVERT": (INVERT, [("", ""), ("0", "1"), ("1", "0"), ("10110", "01001"), ("000", "111")]),
    "ADD": (ADD, [("1+1", "11"), ("111+11", "11111"), ("+1", "1"), ("11+", "11"), ("+", ""), ("1111+111", "1111111")]),
    "INC": (INC, [("0", "1"), ("1", "10"), ("1011", "1100"), ("111", "1000"), ("1001", "1010"), ("110", "111")]),
}
start_checks()
for name, (table, cases) in TESTS.items():
    for word, expected in cases:
        got = safe(run_turing, table, word)
        check(f"{name}('{word}') -> '{expected}'", got[0] if isinstance(got, tuple) else got, expected)
finish_checks()

**1.10. Вывести трассировку работы `INC` на слове `1011`.**

1) Каждая конфигурация выводится отдельной строкой в формате `шаг | состояние | позиция головки | лента`. 
2) После трассировки вывести результат и число шагов.

In [ ]:
output = safe(run_turing, INC, "1011")
result, steps, trace = output if isinstance(output, tuple) and len(output) == 3 else (None, 0, [])
if result is None:
    print("Функция run_turing пока не возвращает результат: выполнить блоки 1.2-1.5")
# ЗАДАНИЕ: вывести строки трассировки и итог (результат и число шагов)

**1.11. Запустить машину на произвольном слове (для самопроверки).**

In [ ]:
TABLES = {"INVERT": INVERT, "ADD": ADD, "INC": INC}
name = input("Таблица (INVERT, ADD, INC): ").strip().upper()
word = input("Слово на ленте: ").strip()
if name in TABLES:
    try:
        output = run_turing(TABLES[name], word)
        if isinstance(output, tuple) and len(output) == 3:
            result, steps, trace = output
            print("Результат:", repr(result), "| шагов:", steps)
            for number, state, head, tape_text in trace:
                print(f"{number:>3} | {state:<5} | {head:>2} | {tape_text}")
        else:
            print("Функция run_turing пока не возвращает результат: выполнить блоки 1.2-1.5")
    except (ValueError, RuntimeError) as error:
        print("Аварийная остановка:", error)
else:
    print("Неизвестная таблица")

**Пояснение к части 1.** Своими словами записать (5-7 предл.): 
1) что входит в конфигурацию машины и как она меняется на шаге;
2) как по трассировке `INC` на слове `1011` видно, откуда берется результат `1100`; 
3) чем нормальная остановка отличается от аварийной и в каких случаях машина останавливается аварийно; 
4) почему таблицу правил можно считать «программой» машины. 

Удалить этот текст и записать пояснение обычным шрифтом.

## Часть 2. ELIZA

**Цель.** Реализовать программу ELIZA (Вейценбаум, 1966) по формальному описанию и написать собственный сценарий.

**Зачем это нужно.** ELIZA - диалоговая система, которая отвечает без понимания смысла: она находит в реплике ключевое слово, подбирает правило и подставляет части фразы в заготовленный ответ. 

Это пример системы, в которой поведение целиком задано правилами. Реализация по частям показывает, из каких шагов складывается внешне осмысленный ответ, и позволяет объяснить, почему программа выдает именно такой ответ и где она ошибается.

**Сценарий.** Сценарий – словарь с четырьмя полями:

| Поле | Содержание |
|---|---|
| `keywords` | словарь ключевых слов: для каждого слова `rank` (ранг; чем больше число, тем выше приоритет) и список `rules` из правил; правило – шаблон разбора `pattern` и список шаблонов ответа `replies` |
| `synonyms` | словарь `слово реплики: ключевое слово` (например, `"мама": "семья"`) |
| `transpose` | словарь замены слов при подстановке в ответ (`"я": "вы"`, `"мой": "ваш"`) |
| `none` | список ответов на случай, когда ни одно правило не подошло |

**Шаблон разбора** – строка из элементов, разделенных пробелами: `0` – любое число слов (в том числе ни одного); `1`, `2`, `3` – ровно столько слов; остальные элементы – сами слова. 

Пример: шаблон `0 помню 0` делит реплику «я помню свое детство» на три части: (1) «я», (2) «помню», (3) «свое детство». 

**Шаблон ответа** содержит ссылки `(1)`, `(2)`, … на части реплики: из `Вы часто вспоминаете (3)?` получается «Вы часто вспоминаете свое детство?».

**Порядок работы программы.**

| Шаг | Действие | Функция |
|---|---|---|
| 1 | Реплика делится на слова и знаки препинания | `tokenize` |
| 2 | Выбирается часть реплики для анализа: встретив знак препинания, программа при уже найденном ключевом слове отбрасывает остаток реплики, иначе отбрасывает прочитанную часть. Слова заменяются синонимами | `select_clause` |
| 3 | Найденные ключевые слова упорядочиваются по убыванию ранга; при равных рангах – по положению слева направо | `order_keywords` |
| 4 | Для ключевого слова по порядку проверяются шаблоны разбора; берется первый подошедший | `match` |
| 5 | Из шаблонов ответа правила выбирается следующий по кругу | `choose_reply` |
| 6 | В шаблон ответа подставляются части реплики с заменой местоимений | `fill_template` |
| 7 | Если ничего не подошло, выбирается ответ из списка `none` | `Eliza.reply` |

**Требования к результату.**

| № | Требование |
|---|---|
| 1 | Реализованы функции из таблицы и класс `Eliza`; все проверки на контрольном сценарии пройдены |
| 2 | `Eliza.reply` сохраняет ход работы в атрибуте `last`: часть реплики, найденные ключевые слова, сработавшее ключевое слово, шаблон разбора, части реплики, шаблон ответа |
| 3 | Эталонный сценарий DOCTOR дает ожидаемые ответы на вопросы общей серии |
| 4 | Составлен собственный сценарий `MY_SCRIPT`, проходящий проверку `check_script` |
| 5 | Составлен собственный пул из 8-12 вопросов; по нему выведен диалог с указанием сработавших правил |
| 6 | Найдены три реплики, на которых ELIZA выдает себя, и указаны причины |
| 7 | Заполнено пояснение к части 2 |

**2.1. Выполнить ячейку с контрольными данными: таблица замены местоимений, знаки-разделители и контрольный сценарий из пяти ключевых слов. Прочитать сценарий.**

In [ ]:
DELIMITERS = {".", ",", ";", ":", "!", "?"}   # знаки, делящие реплику на части

# Подстановка местоимений: слово реплики -> слово ответа
TRANSPOSE = {
    "я": "вы", "мне": "вам", "меня": "вас", "мной": "вами",
    "мой": "ваш", "моя": "ваша", "мое": "ваше", "мои": "ваши", "моего": "вашего",
    "моей": "вашей", "моих": "ваших", "моим": "вашим", "мою": "вашу", "моему": "вашему",
    "вы": "я", "вам": "мне", "вас": "меня", "ваш": "мой", "ваша": "моя", "ваше": "мое", "ваши": "мои",
    "мы": "вы", "нам": "вам", "нас": "вас",
    "чувствую": "чувствуете", "хочу": "хотите", "могу": "можете", "думаю": "думаете",
    "знаю": "знаете", "боюсь": "боитесь", "люблю": "любите", "помню": "помните",
    "понимаю": "понимаете", "считаю": "считаете", "работаю": "работаете", "учусь": "учитесь",
    "живу": "живете", "пытаюсь": "пытаетесь", "буду": "будете", "сдам": "сдадите",
    "умею": "умеете", "вспоминаю": "вспоминаете", "мечтаю": "мечтаете",
    "устал": "устали", "устала": "устали", "был": "были", "была": "были",
    "хотел": "хотели", "хотела": "хотели", "спросил": "спросили", "спросила": "спросили",
    "сказал": "сказали", "сказала": "сказали", "забыл": "забыли", "забыла": "забыли",
    "решил": "решили", "решила": "решили",
}

# Контрольный сценарий: фрагмент DOCTOR из пяти ключевых слов (используется в проверках)
TEST_SCRIPT = {
    "none": ["Продолжайте.", "Расскажите об этом подробнее.", "Что это значит для вас?"],
    "transpose": TRANSPOSE,
    "synonyms": {   # слово реплики -> ключевое слово сценария
        "приснился": "сон", "приснилась": "сон", "снится": "сон", "сны": "сон", "сне": "сон",
        "мама": "семья", "мамы": "семья", "маме": "семья", "мать": "семья", "отец": "семья", "папа": "семья",
        "простите": "извините",
    },
    "keywords": {
        "извините": {"rank": 0, "rules": [
            {"pattern": "0 извините 0",
             "replies": ["Извиняться не нужно.", "Здесь не за что просить прощения.",
                         "Что вы чувствуете, когда извиняетесь?"]},
        ]},
        "если": {"rank": 3, "rules": [
            {"pattern": "0 если 0",
             "replies": ["Вы действительно считаете, что (3)?", "Как вы думаете, вероятно ли, что (3)?",
                         "Что произойдет, если (3)?"]},
        ]},
        "сон": {"rank": 4, "rules": [
            {"pattern": "0 сон 0",
             "replies": ["Что подсказывает вам этот сон?", "Часто ли вам снятся такие сны?",
                         "Как этот сон связан с вашей жизнью?"]},
        ]},
        "помню": {"rank": 5, "rules": [
            {"pattern": "0 помню 0",
             "replies": ["Вы часто вспоминаете (3)?", "Что еще вам вспоминается?",
                         "Почему вам вспомнилось (3) именно сейчас?"]},
        ]},
        "семья": {"rank": 10, "rules": [
            {"pattern": "0 семья 0",
             "replies": ["Расскажите подробнее о вашей семье.", "Как вы относитесь к своей семье?",
                         "Кто еще из близких занимает ваши мысли?"]},
        ]},
    },
}

**2.2. Реализовать функцию `tokenize(text)`.**

1) Вход: строка. 
2) Выход: список токенов. 
3) Слова приводятся к нижнему регистру, буква `ё` заменяется на `е`. 
4) Слово – последовательность букв (русских и латинских), цифр и дефисов. 
5) Каждый знак из `. , ; : ! ?` – отдельный токен. 
6) Остальные символы игнорируются.

In [ ]:
def tokenize(text):
    # ЗАДАНИЕ: разбить реплику на токены
    return None

In [ ]:
start_checks()
check("слова и знаки", safe(tokenize, "Привет, Ёлка!"), ["привет", ",", "елка", "!"])
check("цифры и вопрос", safe(tokenize, "Сколько будет 17 плюс 25?"), ["сколько", "будет", "17", "плюс", "25", "?"])
check("посторонние символы", safe(tokenize, "«Да», нет..."), ["да", ",", "нет", ".", ".", "."])
check("пустая строка", safe(tokenize, "   "), [])
finish_checks()

**2.3. Реализовать функцию `select_clause(tokens, keywords, synonyms)`.**

1) Вход: список токенов, словарь ключевых слов, словарь синонимов. 
2) Выход: пара списков `(original, canon)`. 
3) `original` – слова выбранной части реплики; `canon` – те же слова после замены синонимами (`canon[i]` – синоним слова `original[i]`, если он есть, иначе само слово). 
4) Токены читаются слева направо; слова добавляются в обе части. 
5) При чтении знака препинания, если среди слов `canon` уже есть ключевое слово, чтение прекращается. 
6) Если ключевого слова среди слов `canon` нет, обе части очищаются, и чтение продолжается. 
7) Знаки препинания в списки не попадают.

In [ ]:
def select_clause(tokens, keywords, synonyms):
    # ЗАДАНИЕ: выбрать часть реплики и заменить синонимы
    return None

In [ ]:
start_checks()
kw, syn = TEST_SCRIPT["keywords"], TEST_SCRIPT["synonyms"]
def clause(text):
    result = safe(select_clause, tokenize(text), kw, syn)
    # допускается любая пара (кортеж или список) из двух списков слов
    if isinstance(result, (list, tuple)) and len(result) == 2:
        return tuple(list(part) for part in result)
    return result
check("ключевое слово до запятой", clause("Извините, но мне приснился странный сон."), (["извините"], ["извините"]))
check("часть без ключевых слов отбрасывается", clause("Привет. Мне приснился сон"), (["мне", "приснился", "сон"], ["мне", "сон", "сон"]))
check("остаток после запятой отбрасывается", clause("Если я не сдам экзамен, то мама расстроится."), (["если", "я", "не", "сдам", "экзамен"], ["если", "я", "не", "сдам", "экзамен"]))
check("нет ключевых слов", clause("Сегодня хорошая погода."), ([], []))
check("без знаков препинания", clause("мама дома"), (["мама", "дома"], ["семья", "дома"]))
finish_checks()

**2.4. Реализовать функцию `order_keywords(canon, keywords)`.**

1) Вход: список слов `canon` и словарь ключевых слов. 
2) Выход: список найденных ключевых слов без повторов. 
3) Порядок – по убыванию ранга. 
4) При равных рангах – по положению первого вхождения в `canon`.

In [ ]:
def order_keywords(canon, keywords):
    # ЗАДАНИЕ: найти ключевые слова и упорядочить по рангу
    return None

In [ ]:
start_checks()
check("ранг важнее положения", safe(order_keywords, ["моя", "семья", "часто", "видит", "странные", "сон"], kw), ["семья", "сон"])
check("одно слово", safe(order_keywords, ["если", "я", "не", "сдам"], kw), ["если"])
check("нет слов", safe(order_keywords, [], kw), [])
check("равные ранги и повторы", safe(order_keywords, ["б", "а", "б"], {"а": {"rank": 1}, "б": {"rank": 1}}), ["б", "а"])
finish_checks()

**2.5. Реализовать функцию `match(pattern, words)`.**

1) Вход: шаблон разбора как список элементов (`"0 помню 0".split()`) и список слов. 
2) Выход: список пар `(начало, конец)` – по одной паре на элемент шаблона, где `words[начало:конец]` – слова, которым соответствует элемент. 
3) Если шаблон не подходит ко всему списку слов целиком, возвращается `None`. 
4) Элемент `0` соответствует любому числу слов (в том числе нулю); при нескольких вариантах выбирается тот, где первый такой элемент берет наименьшее число слов. 
5) Элемент `n > 0` соответствует ровно `n` словам. 
6) Остальные элементы должны совпасть со словом.

Подсказка: удобно решать рекурсивно, сопоставляя первый элемент шаблона и передавая дальше остаток шаблона и остаток слов; допускается любой другой способ, например преобразование шаблона в регулярное выражение.

In [ ]:
def match(pattern, words):
    # ЗАДАНИЕ: сопоставить шаблон разбора со словами
    return None

In [ ]:
start_checks()
def m(pattern, words):
    result = safe(match, pattern, words)
    # допускается любой список пар (кортежи или списки)
    if isinstance(result, list) and all(isinstance(p, (list, tuple)) for p in result):
        return [tuple(p) for p in result]
    return result
check("0 помню 0", m("0 помню 0".split(), ["я", "помню", "свое", "детство"]), [(0, 1), (1, 2), (2, 4)])
check("пустые части", m("0 помню 0".split(), ["помню"]), [(0, 0), (0, 1), (1, 1)])
check("слова нет", m("0 помню 0".split(), ["я", "знаю"]), None)
check("ровно два слова", m("2 хочу 0".split(), ["я", "очень", "хочу", "спать"]), [(0, 2), (2, 3), (3, 4)])
check("ровно два слова: не хватает", m("2 хочу 0".split(), ["я", "хочу"]), None)
check("длинный шаблон", m("0 я не могу 0".split(), ["я", "не", "могу", "спать"]), [(0, 0), (0, 1), (1, 2), (2, 3), (3, 4)])
check("первое вхождение", m("0 сон 0".split(), ["сон", "про", "сон"]), [(0, 0), (0, 1), (1, 3)])
finish_checks()

**2.6. Реализовать функцию `fill_template(template, parts, transpose)`.**

1) Вход: шаблон ответа, список частей реплики (строки; часть с номером 1 – `parts[0]`), словарь замены слов. 
2) Выход: строка. 
3) Каждая ссылка `(n)` заменяется на часть с номером `n`. 
4) В подставляемой части каждое слово заменяется по словарю `transpose`; слова, которых в словаре нет, остаются. 
5) Ссылка на несуществующую часть заменяется пустой строкой. 
6) Подряд идущие пробелы заменяются одним. 
7) Пробелы перед знаками `? . ! ,` удаляются. 
8) Пробелы по краям отбрасываются.

In [ ]:
def fill_template(template, parts, transpose):
    # ЗАДАНИЕ: подставить части реплики в шаблон ответа
    return None

In [ ]:
start_checks()
check("подстановка части", safe(fill_template, "Вы часто вспоминаете (3)?", ["я", "помню", "свое детство"], TRANSPOSE), "Вы часто вспоминаете свое детство?")
check("замена местоимений", safe(fill_template, "Вы действительно считаете, что (3)?", ["", "если", "я не сдам экзамен"], TRANSPOSE), "Вы действительно считаете, что вы не сдадите экзамен?")
check("шаблон без ссылок", safe(fill_template, "Продолжайте.", [], TRANSPOSE), "Продолжайте.")
check("пустая часть", safe(fill_template, "Почему вы не можете (2)?", ["я не могу", ""], TRANSPOSE), "Почему вы не можете?")
check("несуществующая часть", safe(fill_template, "Почему (7)?", ["а"], TRANSPOSE), "Почему?")
finish_checks()

**2.7. Реализовать функцию `choose_reply(counters, label, replies)`.**

1) Назначение: циклический выбор ответа. 
2) Вход: словарь счетчиков (изменяется на месте), метка правила, список шаблонов ответа. 
3) Выход: очередной шаблон ответа этого правила. 
4) При первом вызове с данной меткой возвращается первый шаблон, при втором – второй и так далее. 
5) После последнего шаблона снова возвращается первый. 
6) Счетчики разных меток независимы.

In [ ]:
def choose_reply(counters, label, replies):
    # ЗАДАНИЕ: выбрать очередной шаблон ответа по кругу
    return None

In [ ]:
start_checks()
counters = {}
check("цикл из трех ответов", [safe(choose_reply, counters, "a", ["x", "y", "z"]) for _ in range(4)], ["x", "y", "z", "x"])
check("независимые метки", safe(choose_reply, counters, "b", ["u", "v"]), "u")
check("счетчик метки a продолжается", safe(choose_reply, counters, "a", ["x", "y", "z"]), "y")
finish_checks()

**2.8. Реализовать метод `Eliza.reply(text)`.**

Класс `Eliza` хранит сценарий `script`, счетчики `counters` и сведения о последнем ответе `last` (уже заданы в `__init__`).

1) Вход: реплика. 
2) Выход: ответ. 
3) Метод выполняет шаги 1-7 из таблицы с помощью функций из блоков 2.2-2.7. 
4) Сценарий не изменяется. 
5) Если в сценарии нет поля `synonyms` или `transpose`, они считаются пустыми. 
6) Перед возвратом ответа в `self.last` записывается словарь с ключами:

| Ключ | Значение |
|---|---|
| `clause` | слова выбранной части реплики (`original`) |
| `found` | упорядоченный список найденных ключевых слов |
| `keyword` | ключевое слово, чье правило сработало (`None`, если ответ из `none`) |
| `pattern` | сработавший шаблон разбора (`None`, если ответ из `none`) |
| `parts` | части реплики по шаблону (пустой список, если ответ из `none`) |
| `template` | выбранный шаблон ответа до подстановки |

7) Метки счетчиков: для правила – пара `(ключевое_слово, номер_правила)`; для ответов из `none` – пара `("none", 0)`.

In [ ]:
class Eliza:
    def __init__(self, script):
        self.script = script
        self.counters = {}
        self.last = {}

    def reply(self, text):
        # ЗАДАНИЕ: получить ответ на реплику и заполнить self.last
        return None

In [ ]:
PHRASES = [
    "Извините, но мне приснился странный сон.",
    "Я помню свое детство.",
    "Если я не сдам экзамен, то мама расстроится.",
    "Моя мама часто видит странные сны.",
    "Сегодня хорошая погода.",
]
EXPECTED = ['Извиняться не нужно.',
 'Вы часто вспоминаете свое детство?',
 'Вы действительно считаете, что вы не сдадите экзамен?',
 'Расскажите подробнее о вашей семье.',
 'Продолжайте.']
start_checks()
for phrase, expected in zip(PHRASES, EXPECTED):
    check(phrase, safe(lambda p: Eliza(TEST_SCRIPT).reply(p), phrase), expected)
bot = Eliza(TEST_SCRIPT)
check("циклический выбор (4 вызова)", [safe(bot.reply, "Мне приснился странный сон.") for _ in range(4)], ['Что подсказывает вам этот сон?',
 'Часто ли вам снятся такие сны?',
 'Как этот сон связан с вашей жизнью?',
 'Что подсказывает вам этот сон?'])
bot = Eliza(TEST_SCRIPT)
safe(bot.reply, "Моя мама часто видит странные сны.")
check("last: найденные слова", bot.last.get("found"), ["семья", "сон"])
check("last: сработавшее слово", bot.last.get("keyword"), "семья")
check("last: шаблон разбора", bot.last.get("pattern"), "0 семья 0")
bot = Eliza(TEST_SCRIPT)
safe(bot.reply, "Я помню свое детство.")
check("last: части реплики", bot.last.get("parts"), ["я", "помню", "свое детство"])
check("last: шаблон ответа", bot.last.get("template"), "Вы часто вспоминаете (3)?")
bot = Eliza(TEST_SCRIPT)
safe(bot.reply, "Сегодня хорошая погода.")
check("last: ответ из none", (bot.last.get("keyword"), bot.last.get("pattern"), bot.last.get("parts")), (None, None, []))
finish_checks()

**2.9. Выполнить ячейку с эталонным сценарием DOCTOR (14 ключевых слов, адаптация сценария из статьи 1966 года) и общей серией вопросов. Проверить реализацию на эталонном сценарии.**

In [ ]:
# Эталонный сценарий DOCTOR (адаптация на русский язык)
DOCTOR = {
    "none": ["Продолжайте.", "Расскажите об этом подробнее.", "Что это значит для вас?",
             "Понимаю. Что было дальше?", "Почему вы так считаете?"],
    "transpose": TRANSPOSE,
    "synonyms": {
        "приснился": "сон", "приснилась": "сон", "приснилось": "сон", "снится": "сон", "снилось": "сон",
        "сны": "сон", "сне": "сон",
        "мама": "семья", "мамы": "семья", "маме": "семья", "маму": "семья", "мамой": "семья",
        "мать": "семья", "отец": "семья", "папа": "семья", "папе": "семья", "брат": "семья",
        "сестра": "семья", "родители": "семья", "родителей": "семья", "семьи": "семья", "семье": "семья",
        "компьютеры": "компьютер", "компьютера": "компьютер", "машина": "компьютер", "машины": "компьютер",
        "машин": "компьютер", "робот": "компьютер", "нейросеть": "компьютер",
        "простите": "извините", "извини": "извините", "привет": "здравствуйте",
        "друзья": "друг", "друзей": "друг", "друга": "друг", "подруга": "друг",
        "вспоминаю": "помню", "постоянно": "всегда",
    },
    "keywords": {
        "компьютер": {"rank": 50, "rules": [
            {"pattern": "0 компьютер 0",
             "replies": ["Почему вы упомянули машины?", "Тревожат ли вас компьютеры?",
                         "Что вы думаете о машинах?"]},
        ]},
        "семья": {"rank": 10, "rules": [
            {"pattern": "0 семья 0",
             "replies": ["Расскажите подробнее о вашей семье.", "Как вы относитесь к своей семье?",
                         "Кто еще из близких занимает ваши мысли?"]},
        ]},
        "помню": {"rank": 5, "rules": [
            {"pattern": "0 помню 0",
             "replies": ["Вы часто вспоминаете (3)?", "Что еще вам вспоминается?",
                         "Почему вам вспомнилось (3) именно сейчас?"]},
        ]},
        "сон": {"rank": 4, "rules": [
            {"pattern": "0 сон 0",
             "replies": ["Что подсказывает вам этот сон?", "Часто ли вам снятся такие сны?",
                         "Как этот сон связан с вашей жизнью?"]},
        ]},
        "если": {"rank": 3, "rules": [
            {"pattern": "0 если 0",
             "replies": ["Вы действительно считаете, что (3)?", "Как вы думаете, вероятно ли, что (3)?",
                         "Что произойдет, если (3)?"]},
        ]},
        "друг": {"rank": 3, "rules": [
            {"pattern": "0 друг 0",
             "replies": ["Расскажите подробнее о ваших друзьях.", "Почему друзья пришли вам на ум?",
                         "Что для вас значит дружба?"]},
        ]},
        "почему": {"rank": 2, "rules": [
            {"pattern": "0 почему 0",
             "replies": ["Почему вы спрашиваете?", "А как вы сами думаете?",
                         "Вас часто интересуют такие вопросы?"]},
        ]},
        "здравствуйте": {"rank": 1, "rules": [
            {"pattern": "0 здравствуйте 0",
             "replies": ["Здравствуйте. О чем вы хотели бы поговорить?", "Рад вас слышать. Что вас привело сюда?",
                         "Добрый день. Расскажите, что вас беспокоит."]},
        ]},
        "всегда": {"rank": 1, "rules": [
            {"pattern": "0 всегда 0",
             "replies": ["Можете привести конкретный пример?", "Всегда ли это так?",
                         "Что именно приходит вам на ум, когда вы так говорите?"]},
        ]},
        "извините": {"rank": 0, "rules": [
            {"pattern": "0 извините 0",
             "replies": ["Извиняться не нужно.", "Здесь не за что просить прощения.",
                         "Что вы чувствуете, когда извиняетесь?"]},
        ]},
        "да": {"rank": 0, "rules": [
            {"pattern": "0 да 0",
             "replies": ["Вы кажетесь вполне уверенным.", "Понимаю.", "Вы согласны с этим безоговорочно?"]},
        ]},
        "нет": {"rank": 0, "rules": [
            {"pattern": "0 нет 0",
             "replies": ["Почему вы говорите «нет»?", "Что стоит за этим «нет»?", "Вы уверены в этом?"]},
        ]},
        "мне": {"rank": 0, "rules": [
            {"pattern": "0 мне 0",
             "replies": ["Почему вам (3)?", "Часто ли вам (3)?", "Как вы думаете, почему вам (3)?"]},
        ]},
        "я": {"rank": 0, "rules": [
            {"pattern": "0 я чувствую 0",
             "replies": ["Как давно вы чувствуете (4)?", "Часто ли вы чувствуете (4)?",
                         "Что вы думаете, когда чувствуете (4)?"]},
            {"pattern": "0 я хочу 0",
             "replies": ["Почему вы хотите (4)?", "Вы давно хотите (4)?",
                         "Что изменится, если вы получите желаемое?"]},
            {"pattern": "0 я не могу 0",
             "replies": ["Почему вы не можете (5)?", "Вы пытались (5)?", "Что мешает вам (5)?"]},
            {"pattern": "0 я боюсь 0",
             "replies": ["Давно ли вы боитесь (4)?", "Что именно пугает вас?",
                         "Что вы делаете, когда боитесь (4)?"]},
            {"pattern": "0 я 0",
             "replies": ["Почему вы говорите, что вы (3)?", "Как вы относитесь к тому, что вы (3)?",
                         "Продолжайте, пожалуйста."]},
        ]},
    },
}

# Общая серия вопросов: используется в частях 2 и 3
QUESTIONS_COMMON = [
    "Здравствуйте. Расскажите, как прошел ваш день.",
    "Что в этом дне удивило вас больше всего?",
    "Сколько будет 17 плюс 25?",
    "Что тяжелее: килограмм пуха или килограмм железа?",
    "Напишите одну строку о дожде.",
    "О чем я спросил вас в самом начале нашего разговора?",
    "Какой предмет в университете вам нравится больше всего и почему?",
    "Что вы сделаете, если опоздаете на важную встречу?",
    "Назовите столицу Франции и одну ее достопримечательность.",
    "Сколько ног у трех собак?",
    "Расскажите о случае, когда вы очень обрадовались.",
    "Вам понравился этот разговор?",
]
print("Ключевых слов в сценарии DOCTOR:", len(DOCTOR["keywords"]))

In [ ]:
EXPECTED_DOCTOR = ['Здравствуйте. О чем вы хотели бы поговорить?',
 'Продолжайте.',
 'Расскажите об этом подробнее.',
 'Что это значит для вас?',
 'Понимаю. Что было дальше?',
 'Почему вы говорите, что вы спросили меня в самом начале нашего разговора?',
 'Почему вы спрашиваете?',
 'Вы действительно считаете, что опоздаете на важную встречу?',
 'Почему вы так считаете?',
 'Продолжайте.',
 'Расскажите об этом подробнее.',
 'Что это значит для вас?']
EXTRA = [
    "Я чувствую себя усталым.",
    "Я не могу спать.",
    "Мне грустно.",
    "Мне приснился сон, и я боюсь.",
    "Моя семья всегда меня поддерживает.",
]
EXPECTED_EXTRA = ['Как давно вы чувствуете себя усталым?',
 'Почему вы не можете спать?',
 'Почему вам грустно?',
 'Что подсказывает вам этот сон?',
 'Расскажите подробнее о вашей семье.']
start_checks()
bot = Eliza(DOCTOR)
for number, (question, expected) in enumerate(zip(QUESTIONS_COMMON, EXPECTED_DOCTOR), 1):
    check(f"вопрос {number}", safe(bot.reply, question), expected)
for phrase, expected in zip(EXTRA, EXPECTED_EXTRA):
    check(phrase, safe(lambda p: Eliza(DOCTOR).reply(p), phrase), expected)
finish_checks()

**2.10. Составить собственный сценарий `MY_SCRIPT` на выбранную тему.**

Сценарий должен соответствовать требованиям:

| № | Требование |
|---|---|
| 1 | от 10 до 15 ключевых слов на выбранную тему |
| 2 | не менее трех разных рангов |
| 3 | не менее трех шаблонов ответа у каждого шаблона разбора |
| 4 | не менее двух шаблонов ответа с подстановкой частей реплики `(1)`, `(2)`, … |
| 5 | не менее четырех ответов в списке `none` |
| 6 | сценарий общий: в шаблонах разбора запрещены цепочки из четырех и более слов подряд, повторяющие вопросы общей серии (проверяется ячейкой 2.11) |

Цель автора сценария: чтобы ответы программы по возможности не отличались от ответов человека (в части 3 они будут оценены судьей). Генерация сценария языковой моделью допускается; сценарий должен быть прочитан и понят полностью. Таблицу замены местоимений `TRANSPOSE` можно расширить своими словами.

In [ ]:
TOPIC = ""   # тема сценария, например: студенческая жизнь
ROLE = ""    # роль собеседника, например: куратор группы

MY_SCRIPT = {
    "none": [
        # ЗАДАНИЕ: не менее четырех универсальных ответов
    ],
    "transpose": dict(TRANSPOSE),   # при необходимости дополнить: MY_SCRIPT["transpose"]["слово"] = "слово"
    "synonyms": {
        # слово реплики: ключевое слово сценария, например "экзамены": "экзамен"
    },
    "keywords": {
        # Образец записи ключевого слова (заменить своими; образец можно удалить)
        "экзамен": {"rank": 6, "rules": [
            {"pattern": "0 экзамен 0",
             "replies": ["Что вас больше всего тревожит в экзаменах?",
                         "Как вы обычно готовитесь к экзамену?",
                         "Что для вас значит хорошо сдать экзамен?"]},
        ]},
        # ЗАДАНИЕ: добавить не менее девяти ключевых слов
    },
}
print("Тема:", TOPIC, "| роль:", ROLE, "| ключевых слов:", len(MY_SCRIPT["keywords"]))

**2.11. Проверить сценарий на соответствие требованиям. Исправлять сценарий в ячейке 2.10 и повторять проверку, пока замечаний не останется.**

In [ ]:
def check_script(script, questions):
    # Проверяет структуру сценария и соответствие требованиям 1-6
    problems = []
    keywords = script.get("keywords", {})
    if not 10 <= len(keywords) <= 15:
        problems.append(f"Ключевых слов {len(keywords)}: требуется от 10 до 15")
    if len({item["rank"] for item in keywords.values()}) < 3:
        problems.append("Требуется не менее трех разных рангов")
    if len(script.get("none", [])) < 4:
        problems.append("В списке none требуется не менее четырех ответов")
    uses_parts = 0
    question_words = [tokenize(q) for q in questions]
    for key, item in keywords.items():
        if not item.get("rules"):
            problems.append(f"У ключевого слова «{key}» нет шаблонов разбора")
        for rule in item.get("rules", []):
            pattern = rule["pattern"].split()
            if key not in pattern:
                problems.append(f"Шаблон «{rule['pattern']}» не содержит ключевое слово «{key}»")
            if len(rule["replies"]) < 3:
                problems.append(f"У шаблона «{rule['pattern']}» меньше трех шаблонов ответа")
            for reply in rule["replies"]:
                for ref in re.findall(r"\((\d+)\)", reply):
                    uses_parts += 1
                    if not 1 <= int(ref) <= len(pattern):
                        problems.append(f"В ответе «{reply}» номер ({ref}) выходит за пределы шаблона «{rule['pattern']}»")
            # Цепочки из четырех и более слов подряд, повторяющие вопросы серии
            run = []
            for element in pattern + ["0"]:
                if element.isdigit():
                    if len(run) >= 4:
                        for words in question_words:
                            if any(words[i:i + len(run)] == run for i in range(len(words))):
                                problems.append(f"Шаблон «{rule['pattern']}» повторяет вопрос серии")
                    run = []
                else:
                    run.append(element)
    if uses_parts < 2:
        problems.append("Требуется не менее двух шаблонов ответа с подстановкой частей фразы")
    return problems


problems = check_script(MY_SCRIPT, QUESTIONS_COMMON)
if problems:
    print("Замечания к сценарию:")
    for p in problems:
        print("-", p)
else:
    print("Замечаний нет: сценарий соответствует требованиям")

**2.12. Составить собственный пул вопросов `MY_QUESTIONS` и вывести диалог.**

1) Составить список `MY_QUESTIONS` из 8-12 вопросов на тему сценария. 
2) Включить вопросы, вызывающие разные ключевые слова. 
3) Включить хотя бы один вопрос с несколькими ключевыми словами разных рангов. 
4) Включить хотя бы один вопрос без ключевых слов. 
5) Включить хотя бы два вопроса, вызывающие одно и то же правило. 
6) Для каждого вопроса вывести три строки: вопрос; ответ; сработавшее ключевое слово и шаблон разбора (из `bot.last`; для ответа из `none` вывести «нет правила»).

In [ ]:
MY_QUESTIONS = [
    # ЗАДАНИЕ: 8-12 вопросов на тему сценария
]
bot = Eliza(MY_SCRIPT)
for question in MY_QUESTIONS:
    # ЗАДАНИЕ: получить ответ и вывести три строки (вопрос; ответ; ключевое слово и шаблон разбора)
    pass

**2.13. Найти три реплики, на которых ELIZA (эталонный сценарий DOCTOR) выдает себя, и указать причины.**

1) Записать три реплики в список `breaking` в ячейке ниже. 
2) Для каждой реплики указать причину. 
3) Выполнить ячейку: она покажет ответы программы.

Типичные причины: нет подходящего ключевого слова; неверная грамматика после замены местоимений; ответ не учитывает предыдущую реплику; ответ не относится к сути вопроса (вычисление, знание о мире); повторение ответов по кругу.

In [ ]:
breaking = [
    ("реплика 1", "причина"),
    ("реплика 2", "причина"),
    ("реплика 3", "причина"),
]
bot = Eliza(DOCTOR)
for phrase, reason in breaking:
    print("Реплика: ", phrase)
    print("ELIZA:   ", bot.reply(phrase))
    print("Причина: ", reason)
    print()

**2.14. Провести диалог с собственной ELIZA на произвольных репликах (для самопроверки). Для завершения ввести слово «стоп».**

In [ ]:
bot = Eliza(MY_SCRIPT)
while True:
    phrase = input("Вы: ").strip()
    if phrase.lower() == "стоп":
        break
    print("ELIZA:", bot.reply(phrase))
    print("   [ключевые слова:", bot.last.get("found"), "| правило:", bot.last.get("pattern"), "| части:", bot.last.get("parts"), "]")

**Пояснение к части 2.** Своими словами записать (6-8 предл.): 
1) как реплика проходит все шаги от `tokenize` до ответа; 
2) на каком шаге и почему учитывается ранг; 
3) что делают запятая и точка; 
4) как работает циклический выбор ответа; 
5) почему ELIZA не понимает смысл и в чем это проявляется на трех найденных репликах; 
6) какие решения приняты при составлении собственного сценария.

Удалить этот текст и записать пояснение обычным шрифтом.

## Часть 3. Тест Тьюринга

**Цель.** Провести упрощенную игру в имитацию: получить ответы человека и двух программ ELIZA на одни и те же вопросы и оценить каждый ответ автоматическим судьей.

**Зачем это нужно.** В 1950 году Тьюринг заменил вопрос «Могут ли машины мыслить?» проверяемой процедурой: судья по тексту ответов определяет, кто отвечал, человек или машина. 

Эта часть превращает процедуру в эксперимент с таблицей данных. 

В теме 4 к таблице применяются метрики качества классификации, а в теме 7 в нее добавляются ответы языковых моделей. 

В оригинальной игре судья – человек. Здесь судья – программа, оценивающая признаки ответа, и в теме 4 будет написан собственный судья.

**Протокол.**

| Элемент | Содержание |
|---|---|
| Пул вопросов | 12 вопросов общей серии (`QUESTIONS_COMMON`) и 3 собственных вопроса студента |
| Источники ответов | `human` – сам студент; `eliza_ref` – ELIZA с эталонным сценарием DOCTOR; `eliza_own` – ELIZA с собственным сценарием `MY_SCRIPT` |
| Условия ответов человека | письменно, своими словами, как в обычной переписке; без поисковых систем и моделей ИИ; без нарочито коротких и нарочито странных ответов |
| Единица данных | один ответ (источник, номер вопроса, вопрос, ответ) |
| Судья | функция `judge`: оценка от 0 до 1 и вердикт «человек» или «машина» |

**Судья.** Для каждого ответа проверяются пять признаков. Сработавшим признакам соответствуют веса. 

Сумма весов, ограниченная отрезком от 0 до 1 и округленная до двух знаков, является оценкой («насколько ответ похож на машинный»). 

Если оценка не ниже порога, вердикт «машина», иначе «человек». Оценка играет роль уверенности судьи.

| Признак | Условие срабатывания | Вес |
|---|---|---|
| обращение | среди слов ответа есть слово из `ADDRESS` либо слово длиной не менее 6 символов, оканчивающееся на одно из `IMPERATIVE_ENDINGS` | 0.4 |
| вопрос | в ответе есть знак `?` | 0.3 |
| число | среди слов вопроса есть слово «сколько», а в ответе нет ни цифры, ни слова из `NUMBER_WORDS` | 0.4 |
| повтор | слова ответа, соединенные пробелами, совпадают со словами одного из предыдущих ответов того же источника | 0.4 |
| первое_лицо | среди слов ответа есть слово из `FIRST_PERSON` (признак человека) | –0.3 |

Слова – токены `tokenize` без знаков препинания.

**Требования к результату.**

| № | Требование |
|---|---|
| 1 | Составлены 3 собственных вопроса и ответы человека на все 15 вопросов пула |
| 2 | Реализованы функции `collect`, `signs`, `judge`, `evaluate`, `summarize`; все проверки пройдены |
| 3 | Реализована сквозная обработка: сбор ответов трех источников, оценка судьей, сводка по источникам |
| 4 | Сохранены файлы `replies_<код>.csv` и `judged_<код>.csv` |
| 5 | Выполнена попытка улучшить собственный сценарий по сводке судьи |
| 6 | Заполнен протокол эксперимента и вывод |

**3.1. Выполнить ячейку с константами судьи.**

In [ ]:
ADDRESS = {"вы", "вас", "вам", "вами", "ваш", "ваша", "ваше", "ваши"}
FIRST_PERSON = {"я", "мне", "меня", "мной", "мой", "моя", "мое", "мои", "мы", "нам", "нас"}
IMPERATIVE_ENDINGS = ("йте", "ьте", "ите")
NUMBER_WORDS = {"ноль", "один", "одна", "два", "две", "три", "четыре", "пять", "шесть", "семь", "восемь",
                "девять", "десять", "одиннадцать", "двенадцать", "двадцать", "тридцать", "сорок", "пятьдесят",
                "шестьдесят", "семьдесят", "восемьдесят", "девяносто", "сто"}
WEIGHTS = {"обращение": 0.4, "вопрос": 0.3, "число": 0.4, "повтор": 0.4, "первое_лицо": -0.3}
THRESHOLD = 0.4
print("Признаки судьи:", list(WEIGHTS), "| порог:", THRESHOLD)

**3.2. Составить три собственных вопроса и написать ответы человека на все вопросы пула.**

1) Составить три собственных вопроса, расширяющих общую серию (например, по теме сценария). 
2) Написать ответы человека на все вопросы пула. 
3) Записать ответы в список `human_answers` строго по порядку вопросов пула: сначала 12 вопросов общей серии, затем 3 собственных. 
4) Условия записи ответов приведены в протоколе.

In [ ]:
OWN_QUESTIONS = [
    "",   # ЗАДАНИЕ: собственный вопрос 1
    "",   # собственный вопрос 2
    "",   # собственный вопрос 3
]
POOL = QUESTIONS_COMMON + OWN_QUESTIONS

human_answers = [
    # ЗАДАНИЕ: ответы человека на вопросы пула по порядку (15 ответов)
]

problems = []
if any(not q.strip() for q in OWN_QUESTIONS) or len(set(POOL)) != len(POOL):
    problems.append("Собственные вопросы должны быть непустыми и не повторять вопросы серии")
if len(human_answers) != len(POOL) or any(not a.strip() for a in human_answers):
    problems.append(f"Требуется {len(POOL)} непустых ответов, записано {len(human_answers)}")
print("Замечаний нет" if not problems else "\n".join(problems))

**3.3. Реализовать функцию `collect(source, questions, respond)`.**

1) Вход: имя источника, список вопросов, функция `respond(question, turn)`. 
2) `respond` возвращает ответ на вопрос с номером `turn` (нумерация с 1). 
3) Выход: список словарей с ключами `source`, `turn`, `question`, `answer`. 
4) По одному словарю на вопрос, в порядке вопросов.

In [ ]:
def collect(source, questions, respond):
    # ЗАДАНИЕ: собрать ответы источника на вопросы
    return None

In [ ]:
start_checks()
check("collect: структура", safe(collect, "demo", ["a", "b"], lambda q, n: f"{q}{n}"),
      [{"source": "demo", "turn": 1, "question": "a", "answer": "a1"}, {"source": "demo", "turn": 2, "question": "b", "answer": "b2"}])
check("collect: пустой список вопросов", safe(collect, "demo", [], lambda q, n: ""), [])
finish_checks()

**3.4. Реализовать функцию `signs(question, answer, previous)`.**

1) Вход: вопрос, ответ, список предыдущих ответов того же источника (строки). 
2) Выход: словарь. 
3) Ключи словаря – признаки из `WEIGHTS` (в том же порядке). 
4) Значения – `True` или `False` по условиям из таблицы признаков.

In [ ]:
def signs(question, answer, previous):
    # ЗАДАНИЕ: определить, какие признаки сработали
    return None

In [ ]:
def fired_names(result):
    return [name for name, on in result.items() if on] if isinstance(result, dict) else result

SIGN_CASES = [
    ("Сколько будет 17 плюс 25?", "42", [], []),
    ("Что вы думаете о погоде?", "Почему вы так считаете?", [], ["обращение", "вопрос"]),
    ("Как дела?", "Расскажите подробнее.", [], ["обращение"]),
    ("Сколько ног у трех собак?", "Не знаю", [], ["число"]),
    ("Сколько ног у трех собак?", "Двенадцать.", [], []),
    ("Что скажете?", "Продолжайте.", ["продолжайте"], ["обращение", "повтор"]),
    ("О чем я спросил вас?", "Вы спросили, как у меня прошел день.", [], ["обращение", "первое_лицо"]),
]
start_checks()
for question, answer, previous, expected in SIGN_CASES:
    result = safe(signs, question, answer, previous)
    check(f"{answer!r}", fired_names(result), expected)
check("порядок ключей", list(safe(signs, "в", "о", [])) if isinstance(safe(signs, "в", "о", []), dict) else None, list(WEIGHTS))
finish_checks()

**3.5. Реализовать функцию `judge(question, answer, previous)`.**

1) Вход: тот же, что у `signs`. 
2) Выход: тройка `(оценка, вердикт, сработавшие)`. 
3) Оценка – сумма весов сработавших признаков, ограниченная отрезком от 0 до 1 и округленная до двух знаков. 
4) Вердикт – «машина», если оценка не ниже `THRESHOLD`, иначе «человек». 
5) Сработавшие – список имен сработавших признаков в порядке `WEIGHTS`. 
6) Функция использует `signs`.

In [ ]:
def judge(question, answer, previous):
    # ЗАДАНИЕ: вычислить оценку, вердикт и список сработавших признаков
    return None

In [ ]:
JUDGE_CASES = [
    (("Сколько будет 17 плюс 25?", "42", []), (0.0, "человек", [])),
    (("Что вы думаете о погоде?", "Почему вы так считаете?", []), (0.7, "машина", ["обращение", "вопрос"])),
    (("О чем я спросил вас?", "Вы спросили, как у меня прошел день.", []), (0.1, "человек", ["обращение", "первое_лицо"])),
    (("Сколько ног у трех собак?", "Не знаю", []), (0.4, "машина", ["число"])),
    (("Что скажете?", "Продолжайте.", ["продолжайте"]), (0.8, "машина", ["обращение", "повтор"])),
    (("Сколько будет 2 плюс 2?", "Почему вы спрашиваете?", ["а"]), (1.0, "машина", ["обращение", "вопрос", "число"])),
]
start_checks()
for args, expected in JUDGE_CASES:
    check(f"{args[1]!r}", safe(judge, *args), expected)
finish_checks()

**3.6. Реализовать функцию `evaluate(rows)`.**

1) Вход: список словарей-ответов (результат `collect`); ответы разных источников могут идти подряд. 
2) Выход: список словарей с ключами `item_id` (строка `<AUTHOR>-<источник>-<номер вопроса>`), `source`, `turn`, `score`, `verdict`, `signs` (имена сработавших признаков через `;`). 
3) Для каждого ответа «предыдущими ответами» считаются ответы того же источника с меньшими номерами вопросов. 
4) Порядок выходных записей совпадает с порядком входных.

In [ ]:
def evaluate(rows):
    # ЗАДАНИЕ: оценить каждый ответ судьей
    return None

In [ ]:
demo_rows = [
    {"source": "s", "turn": 1, "question": "Что скажете?", "answer": "Продолжайте."},
    {"source": "s", "turn": 2, "question": "Что скажете?", "answer": "Продолжайте."},
    {"source": "t", "turn": 1, "question": "Сколько будет 2 плюс 2?", "answer": "4"},
]
res = safe(evaluate, demo_rows)
ok = isinstance(res, list) and len(res) == 3
start_checks()
check("evaluate: число записей", len(res) if ok else res, 3)
check("evaluate: код записи", res[0]["item_id"] if ok else res, f"{AUTHOR}-s-1")
check("evaluate: первый ответ", (res[0]["score"], res[0]["verdict"], res[0]["signs"]) if ok else res, (0.4, "машина", "обращение"))
check("evaluate: повтор учитывает только свой источник", (res[1]["score"], res[1]["signs"]) if ok else res, (0.8, "обращение;повтор"))
check("evaluate: другой источник", (res[2]["score"], res[2]["verdict"], res[2]["signs"]) if ok else res, (0.0, "человек", ""))
finish_checks()

**3.7. Реализовать функцию `summarize(judged)`.**

1) Вход: результат `evaluate`. 
2) Выход: словарь `{источник: {"n": число ответов, "mean_score": средняя оценка, "share_machine": доля вердиктов «машина»}}`. 
3) Показатели `mean_score` и `share_machine` округляются до двух знаков. 
4) Порядок источников – по первому появлению.

In [ ]:
def summarize(judged):
    # ЗАДАНИЕ: свести результаты по источникам
    return None

In [ ]:
demo_judged = [
    {"source": "a", "score": 0.4, "verdict": "машина"}, {"source": "a", "score": 0.0, "verdict": "человек"},
    {"source": "b", "score": 0.7, "verdict": "машина"},
]
start_checks()
check("summarize", safe(summarize, demo_judged),
      {"a": {"n": 2, "mean_score": 0.2, "share_machine": 0.5}, "b": {"n": 1, "mean_score": 0.7, "share_machine": 1.0}})
finish_checks()

**3.8. Выполнить эксперимент.**

1) Собрать ответы трех источников на вопросы пула с помощью `collect`. 
2) Для каждой ELIZA создать новый объект `Eliza`, который получает вопросы по порядку одним диалогом. 
3) Ответы человека взять из `human_answers`. 
4) Оценить ответы функцией `evaluate`. 
5) Получить сводку функцией `summarize`. 
6) Вывести сводку таблицей: источник, число ответов, средняя оценка, доля вердиктов «машина». 
7) Сохранить результаты в переменных `rows`, `judged`, `summary`.

In [ ]:
# ЗАДАНИЕ: собрать rows (три источника), judged = evaluate(rows), summary = summarize(judged), вывести таблицу
rows = []
judged = []
summary = {}

**3.9. Сохранить результаты в файлы.**

In [ ]:
REPLIES_FILE = DATA / f"replies_{AUTHOR}.csv"
JUDGED_FILE = DATA / f"judged_{AUTHOR}.csv"

if rows and judged:
    with open(REPLIES_FILE, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["source", "turn", "question", "answer"])
        writer.writeheader()
        writer.writerows(rows)
    with open(JUDGED_FILE, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["item_id", "source", "turn", "score", "verdict", "signs"])
        writer.writeheader()
        writer.writerows(judged)
    print("Файлы:", REPLIES_FILE.name, "|", JUDGED_FILE.name, "| строк:", len(rows), "и", len(judged))
else:
    print("Файлы не сохранены: выполнить шаг 3.8")

**3.10. Найти ответы, на которых судья ошибся.**

1) Найти ответы человека с вердиктом «машина» и ответы собственной ELIZA с вердиктом «человек». 
2) Вывести для каждого вопрос, ответ, оценку и сработавшие признаки.

In [ ]:
# ЗАДАНИЕ: вывести ошибки судьи (используются rows и judged)

**3.11. Улучшить собственный сценарий.**

1) Изменить один элемент сценария в ячейке 2.10 (новое ключевое слово, шаблон разбора или шаблоны ответа) так, чтобы снизить долю вердиктов «машина» для `eliza_own`. 
2) Заново выполнить ячейки 2.11, 3.8-3.10. 
3) Сравнить сводки до и после. 
4) Веса и порог судьи не менять. 
5) Сценарий должен остаться общим (проверка 2.11 проходит). 
6) Записать, что изменено и как изменилась сводка.

**Результат попытки.** Удалить этот текст и записать: что изменено в сценарии, значения доли вердиктов «машина» для `eliza_own` до и после, объяснение результата.

**3.12. Оценить пару «вопрос – ответ» судьей, а также получить ответы трех источников на случайный вопрос пула (для самопроверки).**

In [ ]:
def show_verdict(question, answer):
    output = safe(judge, question, answer, [])
    if isinstance(output, tuple) and len(output) == 3:
        score, verdict, fired = output
        print(f"   оценка {score}, вердикт {verdict}, признаки: {fired or 'нет'}")
    else:
        print("   функция judge пока не возвращает результат: выполнить блоки 3.3-3.5")

question = input("Вопрос (Enter – случайный вопрос из пула): ").strip() or random.choice(POOL)
print("Вопрос:", question)
sources = {"eliza_ref": Eliza(DOCTOR), "eliza_own": Eliza(MY_SCRIPT)}
for name, bot in sources.items():
    answer = bot.reply(question)
    print(f"[{name}] {answer}")
    show_verdict(question, answer)
answer = input("Ответ для оценки судьей (Enter – пропустить): ").strip()
if answer:
    show_verdict(question, answer)

**3.13. Заполнить протокол эксперимента: заменить прочерки в правом столбце своими данными.**

| Пункт протокола | Содержание |
|---|---|
| Цель эксперимента | – |
| Источники ответов и число ответов от каждого | – |
| Условия записи ответов человека | – |
| Пул вопросов (общая серия и собственные вопросы) | – |
| Тема и роль собственной ELIZA; число ключевых слов и диапазон рангов | – |
| Судья: признаки, веса, порог | – |
| Результат: доля вердиктов «машина» для каждого источника | – |
| Известные ограничения эксперимента | – |

**3.14. Написать вывод (5-7 предл.).**

1) Вывод пишется безличными предложениями (без «я» и «мы»). 
2) Указать, по каким признакам ответы ELIZA отличались от ответов человека. 
3) Указать, что в собственном сценарии удалось и что не удалось. 
4) Описать ограничения судьи, основанного на признаках, и признаки, которые могли бы улучшить его работу.

**Вывод.** Удалить этот текст и записать вывод обычным шрифтом.

**Пояснение к части 3.** Своими словами записать (5-7 предл.): 
1) как устроена цепочка «пул вопросов – источники – оценка судьи – сводка»; 
2) как из признаков получается оценка и вердикт; 
3) почему признак «повтор» учитывает только ответы того же источника; 
4) на каком примере из шага 3.10 видно ограничение судьи.

Удалить этот текст и записать пояснение обычным шрифтом.

## Справочные ресурсы

| Ресурс | Для чего | Адрес |
|---|---|---|
| Документация Python: модуль re | регулярные выражения | https://docs.python.org/3/library/re.html |
| Документация Python: модуль csv | чтение и запись файлов CSV | https://docs.python.org/3/library/csv.html |
| Документация Python: модуль hashlib | вычисление кода автора | https://docs.python.org/3/library/hashlib.html |
| Основы Markdown | оформление текстовых ячеек | https://www.markdownguide.org/basic-syntax/ |